In [ ]:
#234567890#234567890#234567890#234567890#234567890#234567890#234567890#23456789
from bertopic import BERTopic
from datasets import load_dataset
from hdbscan import HDBSCAN
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from umap import UMAP

In [2]:
dataset = load_dataset('maartengr/arxiv_nlp')['train']
abstracts = dataset['Abstracts']
titles = dataset['Titles']

In [ ]:
embedding_mod = SentenceTransformer('thenlper/gte-small')
embeddings = embedding_mod.encode(abstracts, show_progress_bar=True)

Batches:   0%|          | 0/1405 [00:00<?, ?it/s]

In [ ]:
embeddings.shape

In [ ]:
umap_mod = UMAP(
    n_components=5, min_dist=0., metric='cosine', random_state=123)
reduced_embeddings = umap_mod.fit_transform(embeddings)

In [ ]:
hdbscan_mod = HDBSCAN(
    min_cluster_size=50, metric='euclidean', cluster_selection_method='eom'
).fit(reduced_embeddings)
clusters = hdbscan_mod.labels_
len(set(clusters))

### Inspecting Clusters

In [ ]:
cluster = 1
for i in np.where(clusters == cluster)[0][:3]:
    print(abstracts[i][:300])
    print()

In [ ]:
reduced_embeddings = UMAP(
    n_components=2, min_dist=0., metric='cosine', random_state=123
).fit_transform(embeddings)

In [ ]:
df = pd.DataFrame(reduced_embeddings, columns=['x', 'y'])
df['title'] = titles
df['cluster'] = [str(c) for c in clusters]

In [ ]:
to_plot = df.loc[df.cluster != '-1', :]
outliers = df.loc[df.cluster == '-1', :]

In [ ]:
plt.scatter(outliers.x, outliers.y, alpha=0.05, s=2, c='grey')
plt.scatter(
    to_plot.x,
    to_plot.y, 
    c=to_plot.cluster.astype(int),
    alpha=0.6,
    s=2,
    cmap='tab20b')
plt.axis('off');

## From Text Clustering to Topic Modeling

In [ ]:
topic_mod = BERTopic(
    embedding_model=embedding_mod,
    umap_model=umap_mod,
    hdbscan_model=hdbscan_mod,
    verbose=True
).fit(asbtracts, embeddings)

In [ ]:
topic_mod.get_topic_info()